In [1]:
# Import essential libraries for text processing, machine learning, and data manipulation
import pandas as pd
import re
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
# Download NLTK resources for tokenization, lemmatization, and grammatical tagging
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

True

In [4]:
# Function to remove emojis using a regular expression that covers most corresponding Unicode ranges
def remove_emojis(text):
    if isinstance(text, str):
        pattern = re.compile("[" 
            "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF" "\U0001F700-\U0001F77F"
            "\U0001F780-\U0001F7FF" "\U0001F800-\U0001F8FF"
            "\U0001F900-\U0001F9FF" "\U0001FA00-\U0001FA6F"
            "\U0001FA70-\U0001FAFF" "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251" "]+", flags=re.UNICODE)
        return pattern.sub(r'', text)
    return text

# Initialize the WordNet-based lemmatizer
lemmatizer = WordNetLemmatizer()

# Auxiliary function to obtain the grammatical tag (POS) of a word, necessary for correct lemmatization
def get_pos_tag(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_type = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_type.get(tag, wordnet.NOUN)  # By default, it is considered a noun

# Main lemmatization function: tokenizes the text and applies lemmatization with POS
def lemmatize_text(text):
    words = nltk.word_tokenize(text)
    return ' '.join([lemmatizer.lemmatize(w, get_pos_tag(w)) for w in words])


# Load the labeled SMS dataset (ham/spam) from a public source
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Convert textual labels to binary values: 'ham' → 0 and 'spam' → 1
df['label'] = df['label'].replace({'ham': 0, 'spam': 1})

# Apply preprocessing: remove emojis and lemmatize the messages
df['message'] = df['message'].apply(remove_emojis).apply(lemmatize_text)

# Define stopwords that will be excluded from the TF-IDF model
stopwords = ['a', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
            'is', 'it', 'for', 'with', 'that', 'this', 'as', 'was', 'be',
            'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
            'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not']

In [5]:
i=0
# Define predictive variables (X) and the target variable (y)
X_text = df['message']
y = df['label']

# Configure stratified cross-validation with 5 folds,
# preserving the proportion of classes in each partition
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

# Main cross-validation loop
for train_idx, test_idx in skf.split(X_text, y):
    # Split the training and test sets
    X_train_text = X_text.iloc[train_idx]
    X_test_text = X_text.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Vectorize the texts using TF-IDF with unigrams and bigrams
    # and a limit of 10,000 features. Fitting is performed only on the training set.
    vectorizer = TfidfVectorizer(max_features=10000, stop_words=stopwords)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    # Initialize and train the XGBoost model with predefined hyperparameters
    model = XGBClassifier(
        n_estimators=500,        # Number of trees in the model
        max_depth=6,             # Maximum depth of each tree
        learning_rate=0.01,      # Learning rate
        random_state=42          # Seed for reproducible results
    )
    model.fit(X_train, y_train)
    print("Iteration: "+str(i) )  # Print progress information for the current iteration


    # Predict probabilities for the positive class (spam) and compute AUC-ROC
    probabilities = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probabilities)
    scores.append(auc)
    print("AUC-ROC: Fold " + str(i) +  " : " + str(auc))  # Print the AUC score for the current fold
    i=i+1

Iteration: 0
AUC-ROC: Fold 0:0.9806632124352331
Iteration: 1
AUC-ROC: Fold 1:0.9831778929188255
Iteration: 2
AUC-ROC: Fold 2:0.9831206314984179
Iteration: 3
AUC-ROC: Fold 3:0.9789825086066001
Iteration: 4
AUC-ROC: Fold 4:0.9868484195152484


In [6]:
# Organize the results
results=[]
results.append({
        'k_components': 10000,
        'mean_auc_roc': np.mean(scores)
    })

df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=10000):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=10000):

 k_components  mean_auc_roc
        10000         0.9826
